GRUPO:


*   ARTUR
*   FAVALESSA


*   MANETTI
*   SCHOWANTZ





#Projeto Transformação de Coordenadas

Transformações de coordenadas têm várias aplicações práticas em eng. cartográfica. Alguns exemplos incluem transformação entre sistemas (geodésicos) de referência e georreferenciamento de imagens orbitais, dentre outros.

1) Coordenadas medidas em uma fotografia aérea precisam ser transformadas para o sistema de coordenadas fiducial da câmera. Na tabela abaixo, considere $x_i$ e $y_i$ as coordenadas (em $mm$) das marcas fiduciais calibradas (constantes); e $X_i$ e $Y_i$ as coordenadas (em $mm$) observadas para as mesmas marcas e para outros pontos. Estimar os parâmetros das transformações afim e ortogonal no plano, a partir dos pontos comuns (fiduciais).

2) Para cada transformação, calcule o erro quadrático médio (EQM) dos pontos fiduciais (1 a 4) e compare os resultados.

3) Aplique as transformações aos demais pontos (5 a 9).

Ponto|Coord. Calibrada $x$|Coord. Calibrada $y$|Coord. Medida $X$|Coord. Medida $Y$
--|--|--|--|--
1| -113.000| 0.000| -112.266| 1.270
2| 113.000| 0.000| 113.869| -0.267
3| 0.000| 113.000| 1.792| 113.804
4| 0.000| -113.000| -0.177| -112.350
5| | |-89.750| -0.906
6| | |-71.918| -37.866
7| | |-11.516| -87.300
8| | |-5.276| -0.572
9| | |-20.588| 71.793



## Solução

In [ ]:
import numpy as np
import pandas as pd

# 1. DADOS DE ENTRADA

# Pontos fiduciais 1 a 4
x_cal = np.array([-113.0, 113.0,   0.0,   0.0])
y_cal = np.array([   0.0,   0.0, 113.0, -113.0])
X_med = np.array([-112.266, 113.869, 1.792, -0.177])
Y_med = np.array([   1.270,  -0.267, 113.804, -112.350])

# Pontos 5–9
X_extra = np.array([-89.750, -71.918, -11.516, -5.276, -20.588])
Y_extra = np.array([-0.906, -37.866, -87.300, -0.572, 71.793])

# 2. AJUSTE DAS TRANSFORMAÇÕES (foto → fiducial)

#  Modelo afim: x = a0 + a1*X + a2*Y ; y = b0 + b1*X + b2*Y
A = np.column_stack((np.ones(len(X_med)), X_med, Y_med))

a_params = np.linalg.inv(A.T @ A) @ (A.T @ x_cal)
b_params = np.linalg.inv(A.T @ A) @ (A.T @ y_cal)

a0, a1, a2 = a_params
b0, b1, b2 = b_params

print(" TRANSFORMAÇÃO AFIM ")
print(f"x = {a0:.6f} + {a1:.6f}·X + {a2:.6f}·Y")
print(f"y = {b0:.6f} + {b1:.6f}·X + {b2:.6f}·Y")

# Resíduos e EQM
x_calc = a0 + a1*X_med + a2*Y_med
y_calc = b0 + b1*X_med + b2*Y_med

vX = x_cal - x_calc
vY = y_cal - y_calc
EQM_afim = np.mean(vX**2 + vY**2)
print(f"EQM (Afim): {EQM_afim:.6f} mm\n")

#  Modelo ortogonal: x = a0 + k*cosθ*X - k*sinθ*Y ; y = b0 + k*sinθ*X + k*cosθ*Y
n = len(X_med)
A_ort = np.zeros((2*n, 4))
L = np.zeros((2*n, 1))

for i in range(n):
    A_ort[2*i,   :] = [1, 0, X_med[i], -Y_med[i]]
    A_ort[2*i+1, :] = [0, 1, Y_med[i],  X_med[i]]
    L[2*i,   0] = x_cal[i]
    L[2*i+1, 0] = y_cal[i]

p = np.linalg.inv(A_ort.T @ A_ort) @ (A_ort.T @ L)
a0, b0, kcos, ksin = p.flatten()
k = np.sqrt(kcos**2 + ksin**2)
theta = np.degrees(np.arctan2(ksin, kcos))

print(" TRANSFORMAÇÃO ORTOGONAL ")
print(f"x = {a0:.6f} + {kcos:.6f}·X - {ksin:.6f}·Y")
print(f"y = {b0:.6f} + {ksin:.6f}·X + {kcos:.6f}·Y")
print(f"Escala k = {k:.8f}")
print(f"Rotação θ = {theta:.6f}°")

x_calc_ort = a0 + kcos*X_med - ksin*Y_med
y_calc_ort = b0 + ksin*X_med + kcos*Y_med
vX_ort = x_cal - x_calc_ort
vY_ort = y_cal - y_calc_ort
EQM_ort = np.mean(vX_ort**2 + vY_ort**2)
print(f"EQM (Ortogonal): {EQM_ort:.6f} mm\n")

# 3. APLICAÇÃO AOS PONTOS

#  Afim
x_extra_afim = a_params[0] + a_params[1]*X_extra + a_params[2]*Y_extra
y_extra_afim = b_params[0] + b_params[1]*X_extra + b_params[2]*Y_extra

#  Ortogonal
x_extra_ort = a0 + kcos*X_extra - ksin*Y_extra
y_extra_ort = b0 + ksin*X_extra + kcos*Y_extra

# 4. RESULTADOS

df_result = pd.DataFrame({
    "Ponto": np.arange(5, 10),
    "x_afim": x_extra_afim,
    "y_afim": y_extra_afim,
    "x_ort": x_extra_ort,
    "y_ort": y_extra_ort
})

print(" RESULTADOS")
print(df_result.to_string(index=False))


 TRANSFORMAÇÃO AFIM 
x = -0.798628 + 0.999344·X + -0.008701·Y
y = -0.619258 + 0.006792·X + 0.999258·Y
EQM (Afim): 0.012702 mm

 TRANSFORMAÇÃO ORTOGONAL 
x = -0.799178 + 0.999299·X - 0.007746·Y
y = -0.620051 + 0.007746·X + 0.999299·Y
Escala k = 0.99932910
Rotação θ = 0.444130°
EQM (Ortogonal): 0.024375 mm

 RESULTADOS
 Ponto     x_afim     y_afim      x_ort      y_ort
     5 -90.481857  -2.134145 -90.479252  -2.220642
     6 -72.339977 -38.945608 -72.373449 -39.016605
     7 -11.547494 -87.932689 -11.630859 -87.948066
     8  -6.066189  -1.226667  -6.067049  -1.232520
     9 -21.997774  70.980638 -21.928874  70.963147
